In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score
from sklearn.model_selection import StratifiedKFold
from pyod.models.deep_svdd import DeepSVDD
import sys
import scrapbook as sb

sys.path.append('..')
from utils import reshape_to_numpy, interpolate_missing_values, denoise_data, compute_spectrograms, time_avg_pooling

In [2]:
seed = 1
sampling_rate = 10
nperseg=64
noverlap=32
accel_cutoff = 0.4
accel_order = 30
gyro_cutoff = 0.6
gyro_order = 20
batch_size = 32
epochs = 100
hidden_neurons = [128, 64, 32]
n_splits = 5

In [3]:
# Parameters
seed = 14


In [4]:
rng = np.random.RandomState(seed)

In [5]:
data = pd.read_parquet("../data/GBG500.parquet")
data

,ride_id,time_index,ax,ay,az,rx,ry,rz
0,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,0,-2.512796,-9.385012,-1.053078,-0.009155,0.009155,-0.201416
1,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,100,-2.491268,-9.385012,-1.079390,-0.036621,0.027465,-0.155639
2,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,200,-2.534324,-9.382022,-1.030952,0.036621,0.027465,-0.073242
3,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,300,-2.488278,-9.383218,-1.091948,-0.045776,0.036621,-0.183105
4,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,400,-2.483494,-9.382022,-1.100320,-0.045776,0.027465,-0.192260
...,...,...,...,...,...,...,...,...
1529542,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241300,0.459862,-9.140430,-3.318900,1.556396,-0.091552,0.274658
1529543,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241400,0.455676,-9.140430,-3.321292,1.583861,-0.119018,0.274658
1529544,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241500,0.459862,-9.139832,-3.317704,1.583861,-0.109863,0.274658
1529545,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241600,0.456274,-9.142224,-3.321292,1.574707,-0.137329,0.283813


In [6]:
labels = pd.read_csv("../data/GBG500_labels.csv")
labels.columns = labels.columns.str.lower()
ride_order_df = pd.DataFrame({"ride_id": data["ride_id"].unique()})
labels_sorted = ride_order_df.merge(
    labels,
    on="ride_id",
    how="left"
)
labels_sorted

,ride_id,label
0,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,Safe
1,00d223cb7aecc9c0cc5871e32a6c45027a068eba07c572...,Reckless
2,02288a4aeca044203394e982e76b87818021dea2b34df9...,Safe
3,0236bdcf13d473ea24d97f5eaeea459f257dffac005694...,Bad weather
4,0238e5dd85143f1b6c59b200bc32f14361b8fc84c51a87...,Safe
...,...,...
495,fe5d7f84d692bbccad8bd566a3ad5c3108fa9f90acd671...,Safe
496,fe67169632306d4668b2affedef510df2cb43e2fb41220...,Safe
497,fee44667fdc8ab4995c702fc3bd36178de0b1ca2c64687...,Safe
498,ff79b2e945b93c2a5efc3a06365267b3a160e7849ba57d...,Safe


In [7]:
data_np = reshape_to_numpy(
    data,
    features = ["ax", "ay", "az", "rx", "ry", "rz"],
    max_timestamps = 4800
)

data_np_clean = interpolate_missing_values(
    data_np,
    method='linear',
    limit=None
)

data_np_clean = denoise_data(
    data=data_np_clean,
    accel_indices=[0, 1, 2],
    gyro_indices=[3, 4, 5],
    accel_cutoff=accel_cutoff,
    accel_order=accel_order,
    gyro_cutoff=gyro_cutoff,
    gyro_order=gyro_order,
)

In [8]:
# Compute spectrograms for all rides
spectrograms_array = compute_spectrograms(data_np_clean, sampling_rate, nperseg, noverlap)
spectrograms_array.shape

(500, 149, 6, 33)

In [9]:
# Aggregate spectrograms using time-averaged pooling
X_feat = time_avg_pooling(spectrograms_array)
X_feat.shape

(500, 198)

In [10]:
# 5-fold stratified CV with Deep SVDD
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=rng.randint(1000))
y_true = (labels_sorted['label'] == 'Reckless').astype(int).values
anomaly_scores = np.full(len(y_true), np.nan)

for fold, (train_idx, test_idx) in enumerate(skf.split(X_feat, y_true)):
    sc = StandardScaler()
    X_train = sc.fit_transform(X_feat[train_idx])
    X_test = sc.transform(X_feat[test_idx])

    np.random.seed(rng.randint(1000))
    model = DeepSVDD(
        n_features=X_train.shape[1],
        hidden_neurons=hidden_neurons,
        epochs=epochs,
        batch_size=batch_size,
        random_state=rng.randint(1000),
    )
    model.fit(X_train)
    anomaly_scores[test_idx] = model.decision_function(X_test)
    print(f"Fold {fold+1}/{n_splits} done")

Epoch 1/100, Loss: 2.4426666498184204
Epoch 2/100, Loss: 2.608601853251457
Epoch 3/100, Loss: 2.5902160704135895
Epoch 4/100, Loss: 2.5618813782930374
Epoch 5/100, Loss: 2.631891056895256
Epoch 6/100, Loss: 2.637956030666828
Epoch 7/100, Loss: 2.616042733192444
Epoch 8/100, Loss: 2.450037993490696
Epoch 9/100, Loss: 2.946452982723713
Epoch 10/100, Loss: 2.6029019877314568
Epoch 11/100, Loss: 2.6086739227175713
Epoch 12/100, Loss: 2.463593363761902
Epoch 13/100, Loss: 2.486408054828644
Epoch 14/100, Loss: 2.7008191496133804


Epoch 15/100, Loss: 2.466580606997013
Epoch 16/100, Loss: 2.6656754687428474
Epoch 17/100, Loss: 2.575024664402008
Epoch 18/100, Loss: 2.5677152797579765
Epoch 19/100, Loss: 2.5226596891880035
Epoch 20/100, Loss: 2.4809425622224808
Epoch 21/100, Loss: 2.6371935531497
Epoch 22/100, Loss: 2.696437254548073
Epoch 23/100, Loss: 2.5583334490656853
Epoch 24/100, Loss: 2.6157729104161263
Epoch 25/100, Loss: 2.6852881386876106


Epoch 26/100, Loss: 2.6538574919104576
Epoch 27/100, Loss: 2.58663809299469
Epoch 28/100, Loss: 2.5691468566656113
Epoch 29/100, Loss: 2.480208098888397
Epoch 30/100, Loss: 2.6003874987363815
Epoch 31/100, Loss: 2.6029820889234543
Epoch 32/100, Loss: 2.5285900309681892
Epoch 33/100, Loss: 2.411407455801964
Epoch 34/100, Loss: 2.6274554058909416
Epoch 35/100, Loss: 2.6335165798664093
Epoch 36/100, Loss: 2.6856530904769897
Epoch 37/100, Loss: 2.531921550631523


Epoch 38/100, Loss: 2.457483872771263
Epoch 39/100, Loss: 2.5966762006282806
Epoch 40/100, Loss: 2.516511969268322
Epoch 41/100, Loss: 2.5109338387846947
Epoch 42/100, Loss: 2.668545126914978
Epoch 43/100, Loss: 2.5622042790055275
Epoch 44/100, Loss: 2.5229716449975967
Epoch 45/100, Loss: 2.6206256076693535
Epoch 46/100, Loss: 2.500844292342663
Epoch 47/100, Loss: 2.6514531821012497
Epoch 48/100, Loss: 2.568036139011383
Epoch 49/100, Loss: 2.5494880378246307
Epoch 50/100, Loss: 2.5031222254037857


Epoch 51/100, Loss: 2.6693407371640205
Epoch 52/100, Loss: 2.5849233344197273
Epoch 53/100, Loss: 2.618251569569111
Epoch 54/100, Loss: 2.3543727099895477
Epoch 55/100, Loss: 2.3260765448212624
Epoch 56/100, Loss: 2.526831716299057
Epoch 57/100, Loss: 2.486838199198246
Epoch 58/100, Loss: 2.651527613401413
Epoch 59/100, Loss: 2.529101699590683
Epoch 60/100, Loss: 2.5535852313041687
Epoch 61/100, Loss: 2.691743016242981
Epoch 62/100, Loss: 2.5449487194418907
Epoch 63/100, Loss: 2.6030604541301727
Epoch 64/100, Loss: 2.616456024348736
Epoch 65/100, Loss: 2.5750406682491302


...

Epoch 30/100, Loss: 2.170343339443207
Epoch 31/100, Loss: 2.216292627155781
Epoch 32/100, Loss: 2.0516935363411903
Epoch 33/100, Loss: 2.1305867582559586
Epoch 34/100, Loss: 2.126267835497856
Epoch 35/100, Loss: 2.167370520532131
Epoch 36/100, Loss: 2.1492114514112473
Epoch 37/100, Loss: 1.9032099172472954
Epoch 38/100, Loss: 1.9987109899520874
Epoch 39/100, Loss: 2.2356559485197067
Epoch 40/100, Loss: 2.353522799909115
Epoch 41/100, Loss: 2.1228377670049667
Epoch 42/100, Loss: 2.217687301337719
Epoch 43/100, Loss: 2.0478165671229362
Epoch 44/100, Loss: 2.066004306077957
Epoch 45/100, Loss: 2.088765934109688


Epoch 46/100, Loss: 2.1438718736171722
Epoch 47/100, Loss: 2.1868271082639694
Epoch 48/100, Loss: 2.009961064904928
Epoch 49/100, Loss: 2.1698475182056427
Epoch 50/100, Loss: 2.328298419713974
Epoch 51/100, Loss: 2.220586620271206
Epoch 52/100, Loss: 1.9495709538459778
Epoch 53/100, Loss: 2.189121685922146
Epoch 54/100, Loss: 2.133744701743126
Epoch 55/100, Loss: 2.270007811486721
Epoch 56/100, Loss: 2.2384551018476486
Epoch 57/100, Loss: 2.143809035420418
Epoch 58/100, Loss: 1.9191867485642433
Epoch 59/100, Loss: 2.096892476081848
Epoch 60/100, Loss: 1.987608090043068
Epoch 61/100, Loss: 2.214928977191448
Epoch 62/100, Loss: 2.52737570554018


Epoch 63/100, Loss: 2.1192757040262222
Epoch 64/100, Loss: 2.0524149164557457
Epoch 65/100, Loss: 2.1129558756947517
Epoch 66/100, Loss: 2.103533662855625
Epoch 67/100, Loss: 2.1681125164031982
Epoch 68/100, Loss: 1.9835579469799995
Epoch 69/100, Loss: 2.040022075176239
Epoch 70/100, Loss: 2.1089195385575294
Epoch 71/100, Loss: 2.2433940693736076
Epoch 72/100, Loss: 2.0917571410536766
Epoch 73/100, Loss: 2.0287961140275
Epoch 74/100, Loss: 1.89654341340065
Epoch 75/100, Loss: 2.0613269731402397
Epoch 76/100, Loss: 2.1451771780848503
Epoch 77/100, Loss: 2.078535333275795
Epoch 78/100, Loss: 2.1128569319844246
Epoch 79/100, Loss: 1.9845228046178818


Epoch 80/100, Loss: 2.258721563965082
Epoch 81/100, Loss: 2.1533425599336624
Epoch 82/100, Loss: 2.0134895890951157
Epoch 83/100, Loss: 1.9264230355620384
Epoch 84/100, Loss: 2.6590181291103363
Epoch 85/100, Loss: 1.9948744177818298
Epoch 86/100, Loss: 2.131309397518635
Epoch 87/100, Loss: 2.0955545976758003
Epoch 88/100, Loss: 2.1932923644781113
Epoch 89/100, Loss: 2.1611455604434013
Epoch 90/100, Loss: 1.9370846524834633
Epoch 91/100, Loss: 2.1689850240945816
Epoch 92/100, Loss: 2.0426853597164154
Epoch 93/100, Loss: 1.9128656163811684
Epoch 94/100, Loss: 2.033068284392357
Epoch 95/100, Loss: 2.2448842003941536
Epoch 96/100, Loss: 1.9683327078819275


Epoch 97/100, Loss: 2.0151682272553444
Epoch 98/100, Loss: 1.8863408118486404
Epoch 99/100, Loss: 2.1496087685227394
Epoch 100/100, Loss: 1.9495989754796028
Fold 5/5 done


In [11]:
ap = average_precision_score(y_true, anomaly_scores)
print(f"Deep SVDD AP (5-fold CV) = {ap:.4f}")
sb.glue("GBG500_ap_spectral_deep_svdd", float(ap))

Deep SVDD AP (5-fold CV) = 0.4886
